<a href="https://colab.research.google.com/github/MahiDeshpande/when-metrics-disagree/blob/main/when_metrics_disagree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **When Metrics Disagree: A Visual Investigation of PSNR, SSIM, LPIPS, and TPIPS**

## **The main question?**
Our main question that we are trying to answer is: What does “image quality” actually mean, and when do PSNR, SSIM, LPIPS, and newer perceptual metrics (TPIPS) agree or disagree with what humans see?

This project explores three core layers:
1. **Understanding Metric Perceptions:** Analyzing how each algorithm interprets image variations.
2. **Controlled Stress-Testing:** Subjecting benchmark images to synthetic transformations (pixel shifts, Gaussian blurs, noise, JPEG compression, and brightness shifts).
3. **Counterexample Discovery:** Finding failure modes where metric score rankings directly contradict human visual intuition.
4. **Bonus:** Going a little more in-depth into different models (SSIM vs MS-SSIM). Researching differences and/or variation when using different backbones (testing out AlexNet vs VGG for LPIPS)
---

## **Progress Tracker**

| Task / Experiment | Test Description | Status |
| :--- | :--- | :---: |
| **Setup & Dependencies** | Import libraries and initialize metric models (`skimage`, `lpips`, `tpips`). | done |
| **Test 1: Spatial Translation** | Evaluate metric sensitivity on 1–5 pixel shifts. | [ ] |
| **Test 2: High-Frequency Blur** | Measure metric response across increasing Gaussian blur radii. | [ ] |
| **Test 3: Noise & Artifacts** | Stress-test with Additive Noise & JPEG compression artifacts. | [ ] |
| **Test 4: Photometric Shifts** | Evaluate uniform brightness and contrast alterations. | [ ] |
| **Counterexample Analysis** | Compile side-by-side failure mode plots showing metric disagreements. | [ ] |

---

## **Extras & Additional Notes**
- **Dataset:** Small image subset provided for benchmark testing.
- **Goal:** Export key visual plots and qualitative findings into the final technical blog post and project repository.

### **Definitions of the 4 metrics we will be comparing** ###

### 1. PSNR (Peak Signal-to-Noise Ratio)

**The Metaphor:** A strict, pixel-by-pixel accountant.
PSNR compares two images by subtracting the exact RGB values of every single pixel in Image A from Image B, squaring the differences, and taking the average (Mean Squared Error). It then converts that error into a logarithmic decibel scale.

* **How it works:** It measures pixel-level reconstruction error. If every pixel matches identically, error is zero, and PSNR is infinite ($\infty$). As pixels differ, PSNR drops.
* **Direction:** **Higher ($\uparrow$) is better.**
* **Typical Range:** $20\text{ dB}$ (heavily degraded) to $40+\text{ dB}$ (virtually identical).
* **The Major Blind Spot ("The Alignment Trap"):** Shift an image to the right by just **2 pixels**. To a human, the image looks completely unchanged. But to PSNR, every single pixel line is now misaligned, causing the score to plummet from $40\text{ dB}$ down to $15\text{ dB}$.

---

### 2. SSIM (Structural Similarity Index)

**The Metaphor:** A localized pattern checker.
Created to fix PSNR's flaws, SSIM doesn't just look at isolated pixels. Instead, it analyzes small local windows (e.g., $8\times 8$ pixel patches) and evaluates three specific features:

1. **Luminance:** Overall brightness of the patch.
2. **Contrast:** Spread of light and dark values within the patch.
3. **Structure:** The directional patterns, textures, and edges.

* **How it works:** It outputs a single normalized score comparing structural correlation between patches.
* **Direction:** **Higher ($\uparrow$) is better.**
* **Typical Range:** $-1.0$ to $1.0$ (where $1.0$ means identical structure).
* **The Major Blind Spot:** SSIM is still bound to spatial coordinates. While it handles mild luminance changes better than PSNR, moderate spatial translations or slight rotations still break the local window comparisons, causing artificial score drops.

---

### 3. LPIPS (Learned Perceptual Image Patch Similarity)

**The Metaphor:** A simulated human visual cortex (CNN feature extractor).
Instead of calculating mathematical distances between pixels, LPIPS passes both images through a pre-trained Convolutional Neural Network (like **AlexNet** or **VGG**).

* **How it works:**
1. The network extracts feature maps across different deep layers (from low-level edges in early layers to high-level shapes in deeper layers).
2. It computes the distance between those deep feature activations.
3. It weights those differences based on human perceptual judgement datasets.


* **Direction:** **Lower ($\downarrow$) is better.**
* **Typical Range:** $0.00$ (perceptually identical) to $\approx 0.70+$ (completely different scene/texture).
* **Why it's better:** If you shift an image by 2 pixels or add mild texture noise, the deep feature activations in AlexNet/VGG remain almost identical. LPIPS correctly reflects that the image still "looks" fine to human eyes.

---

### 4. TPIPS (Text-Prompted Image Perceptual Similarity)

**The Metaphor:** An AI visual-language critic (VLM reasoning model).
TPIPS is the newest metric on the block. Instead of relying on traditional CNNs, it routes images through a massive Vision-Language Model (specifically built on top of backbones like **Qwen3-VL-8B**).

* **How it works:**
1. Images are patchified into multimodal embeddings that link visual concepts directly to text.
2. You can pass a **text prompt condition** (like `factor="overall"`, `factor="lighting"`, or `factor="color"`).
3. The text prompt acts as an attention steering vector, forcing the network to evaluate perceptual similarity *specifically along that visual dimension*.


* **Direction:** **Lower ($\downarrow$) is better.**
* **Typical Range:** $0.00$ (identical semantic alignment) to $2.0$ (maximum vector distance).
* **Why it's revolutionary:** It understands high-level semantic context. If an image is dark but the object structure, geometry, and key semantic features remain intact, TPIPS evaluates the image with broad context rather than getting tripped up by raw pixel variance.

---

### Metric Comparison Cheatsheet

| Metric | Category | Primary Focus | Unit / Scale | Sensitivity to 2-Pixel Shift |
| --- | --- | --- | --- | --- |
| **PSNR** | Classical Math | Pixel-level RGB error | Decibels ($\text{dB}$) $[0, \infty)$ | **Extreme penalty** (Plummets) |
| **SSIM** | Classical Pattern | Local luminance, contrast, edges | Index $[-1.0, 1.0]$ | **Moderate penalty** |
| **LPIPS** | Deep Features (CNN) | Mid-to-high level feature maps | Distance $[0.0, \approx 1.0]$ | **Invariant** (Stays flat) |
| **TPIPS** | Vision-Language (VLM) | High-level scene & aspect semantics | Distance $[0.0, 2.0]$ | **Invariant** (Stays flat) |

## **Installing Required Libraries To Run Image Quality Metrics**

OpenCV, scikit-image, PyTorch, LPIPS, Matplotlib, and HuggingFace Transfomers (for TPIPS backbones)

---

In [1]:
# -------------------------------------------------------------------------
# NO-TOKEN, NO-CRASH SETUP SCRIPT
# -------------------------------------------------------------------------
!pip install -q --upgrade torchao
!pip install -q opencv-python-headless scikit-image torch torchvision lpips matplotlib ftfy regex tqdm transformers accelerate
!pip install -q git+https://github.com/adobe-research/TPIPS.git

import gc
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import lpips
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import tpips

# ==========================================
# 1. WIPE GHOST MEMORY
# ==========================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. LOAD LPIPS TO GPU
# ==========================================
print("Loading LPIPS to GPU...")
lpips_alex = lpips.LPIPS(net="alex").to(device).eval()
lpips_vgg = lpips.LPIPS(net="vgg").to(device).eval()

# ==========================================
# 3. LOAD TPIPS TO CPU (Bypassing GPU Limits)
# ==========================================
print("Loading TPIPS to CPU (This might take a minute to download)...")

# Compress the model by 50% right as it downloads
torch.set_default_dtype(torch.bfloat16)

# Load safely to system RAM instead of GPU
tpips_model = tpips.load_model("embedding", device="cpu")

# Restore standard PyTorch settings
torch.set_default_dtype(torch.float32)
tpips_model.eval()

print("\n✅ WE MADE IT! All models loaded cleanly.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Loading LPIPS to GPU...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 131MB/s]


...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

[embedding] LoRA on LLM (r=16, alpha=32); 38.9M trainable / 8183.7M total


[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--sywang--TPIPS-Embed-Qwen3VL-8B/snapshots/86a1493c60121e656a1c075bc7aed88f25fcd4d4' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[load_model] embedding loaded from /root/.cache/huggingface/hub/models--sywang--TPIPS-Embed-Qwen3VL-8B/snapshots/86a1493c60121e656a1c075bc7aed88f25fcd4d4 on cpu

✅ WE MADE IT! All models loaded cleanly.



### The Main Functions Used to Carry Out the 4 Metrics and Visualise Them

This block defines our central helper functions (`compute_all_metrics` and `plot_sweep_grid`) that power every test case in this notebook.

To make these 4 fundamentally different metric families **directly comparable**, our pipeline forces every model to evaluate the exact same image pair simultaneously under strict resolution, color space, and memory alignment standards.

**How We Ensure Rigorous Cross-Metric Comparability:**

1. **Input Normalization Standard:** Every input image pair ($I_{\text{ref}}, I_{\text{dist}}$) is passed as a $512 \times 512 \times 3$ NumPy array scaled strictly to $[0.0, 1.0]$. This eliminates resolution bias across model backbones.
2. **Parallel Input Formatting:**
* **Classical Math Metrics (PSNR, SSIM):** Evaluated directly on the raw $[0.0, 1.0]$ float arrays across all 3 color channels ($H \times W \times C$).
* **Deep Feature Models (LPIPS Alex & VGG):** Transformed into 4D PyTorch tensors ($1 \times C \times H \times W$), rescaled to $[-1.0, 1.0]$, and loaded directly onto the GPU (`device`).
* **Multimodal Vision-Language Model (TPIPS):** Converted back into 8-bit PIL images ($0\text{--}255$), maintaining spatial resolution while executing through its deep vision-language embedding space.


3. **Execution & Direction Alignment:** Every metric runs inside inference mode (`torch.no_grad()` / `torch.inference_mode()`) to freeze network weights and prevent gradient accumulation. Scores are recorded alongside directional arrows—**Higher ($\uparrow$)** for quality/similarity, **Lower ($\downarrow$)** for perceptual distance—allowing us to track score trajectory convergence and divergence across experiments.


In [3]:
# =========================================================================
# 2. CORE EVALUATION & VISUALIZATION HELPERS
# =========================================================================

def compute_all_metrics(img_ref_np, img_dist_np, factor="overall"):
    """Computes PSNR, SSIM, LPIPS (Alex & VGG), and TPIPS for an image pair.

    Standardizes inputs across all 4 metric backbones to ensure direct comparability.
    Expects float NumPy arrays in shape (512, 512, 3) with values in [0.0, 1.0].
    """
    # 1. PSNR & SSIM (Classical Metrics at full 512x512 resolution)
    score_psnr = compute_psnr(img_ref_np, img_dist_np, data_range=1.0)
    score_ssim = compute_ssim(
        img_ref_np, img_dist_np, channel_axis=2, data_range=1.0
    )

    # 2. LPIPS (GPU Tensors at full 512x512 resolution)
    tensor_ref = (
        torch.tensor(img_ref_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
        * 2
        - 1
    )
    tensor_dist = (
        torch.tensor(img_dist_np)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .float()
        .to(device)
        * 2
        - 1
    )

    with torch.no_grad():
        score_lpips_alex = lpips_alex(tensor_ref, tensor_dist).item()
        score_lpips_vgg = lpips_vgg(tensor_ref, tensor_dist).item()

    # 3. TPIPS (Optimized: Downsample PIL images to 256x256 for fast VLM inference)
    img_ref_pil = Image.fromarray((img_ref_np * 255).astype(np.uint8)).resize(
        (256, 256)
    )
    img_dist_pil = Image.fromarray((img_dist_np * 255).astype(np.uint8)).resize(
        (256, 256)
    )

    with torch.inference_mode():
        score_tpips = tpips_model.distance(
            img_ref_pil, img_dist_pil, factor=factor
        )
        if isinstance(score_tpips, torch.Tensor):
            score_tpips = score_tpips.item()

    return {
        "PSNR (↑)": round(score_psnr, 2),
        "SSIM (↑)": round(score_ssim, 4),
        "LPIPS-Alex (↓)": round(score_lpips_alex, 4),
        "LPIPS-VGG (↓)": round(score_lpips_vgg, 4),
        "TPIPS Distance (↓)": round(score_tpips, 4),
    }


def plot_sweep_grid(
    img_ref, distortion_fn, levels, param_name="Level", exp_title="Sweep"
):
    """Generates a 2-row layout (Top: Reference, Bottom: Distortions) followed by
    a metric summary table.
    """
    num_levels = len(levels)
    fig, axes = plt.subplots(
        2, num_levels, figsize=(3.5 * num_levels, 6), squeeze=False
    )

    results = []

    for i, lvl in enumerate(levels):
        img_dist = distortion_fn(img_ref, lvl)

        # Top Row: Reference Image
        axes[0, i].imshow(img_ref)
        axes[0, i].set_title("Original Ref", fontsize=10)
        axes[0, i].axis("off")

        # Bottom Row: Distorted Image
        axes[1, i].imshow(img_dist)
        axes[1, i].set_title(f"{param_name}: {lvl}", fontsize=10)
        axes[1, i].axis("off")

        # Compute Metrics
        scores = compute_all_metrics(img_ref, img_dist)
        scores[param_name] = lvl
        results.append(scores)

    plt.suptitle(f"Distortion Grid: {exp_title}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    df_results = pd.DataFrame(results)
    cols = [param_name] + [c for c in df_results.columns if c != param_name]
    display(df_results[cols])

    return df_results

print("✅ Master helper functions and optimized metric pipeline are ready!")

✅ Master helper functions and optimized metric pipeline are ready!


### Phase 1: Experiment 1 — The Basic Distortion Zoo

This phase establishes **Experiment 1**, the foundational stress-testing suite in our comparative Image Quality Assessment (IQA) research framework. The primary objective is to systematically isolate and measure how classical mathematical metrics (**PSNR, SSIM**), deep feature extractors (**LPIPS-Alex, LPIPS-VGG**), and multimodal vision-language models (**TPIPS** running on Qwen3-VL-8B) behave under synthetic, controlled image degradations.

Experiment 1 serves as the entry point to a multi-stage experimental roadmap (comprising higher-order evaluations like Iso-PSNR Calibration, Heatmap Clustering, Rank Disagreement Mining, and Human Perception Alignment).

---

#### 1. Experimental Breakdown (10 Isolated Tests)

Experiment 1 is partitioned into **10 distinct parametric test cells**. Each test isolates a single geometric, noise, or photometric transformation across 4 progressive severity levels:

| Test # | Distortion Category | Parameter Sweep Levels | Primary Geometric / Perceptual Target |
| :--- | :--- | :--- | :--- |
| **Test 1** | **(Clipped) Gaussian Noise** | $\sigma = 5, 10, 20, 40$ | High-frequency additive noise & boundary clipping effects |
| **Test 2** | **Gaussian Blur** | Radius $= 1, 2, 4, 8\text{ px}$ | High-frequency loss & smooth detail suppression |
| **Test 3** | **JPEG Compression** | Quality $= 90, 70, 40, 10$ | Blocky $8\times8$ frequency artifacts & ringing |
| **Test 4** | **Brightness Shift** | Scaling $= 0.7, 0.9, 1.1, 1.3$ | Global intensity shifts ($\pm 10\%, \pm 30\%$) |
| **Test 5** | **Contrast Alteration** | Scaling $= \times 0.7, \times 1.3$ | Dynamic range expansion and compression |
| **Test 6** | **Saturation Alteration** | Scaling $= \times 0.0, \times 0.5, \times 1.5$ | Color fidelity (Monochrome to over-saturated) |
| **Test 7** | **Spatial Translation** | Shift $= 1, 2, 4, 8\text{ pixels}$ | Spatial alignment sensitivity ("The Alignment Trap") |
| **Test 8** | **Spatial Rotation** | Angle $= 1^\circ, 3^\circ, 5^\circ, 10^\circ$ | Rotational invariance & resampling artifacts |
| **Test 9** | **Crop + Resize** | Crop $= 2\%, 5\%, 10\%, 20\%$ | Field-of-view alteration & interpolation scaling |
| **Test 10** | **Color Shift (Hue)** | Offset $= 10^\circ, 30^\circ, 60^\circ, 90^\circ$ | HSV channel phase rotation |

---

#### 2. Visual & Analytical Output Format

To maintain complete visual clarity and consistency across all 10 notebook cells, every test executes via our unified helper pipeline (`plot_sweep_grid`) and renders a three-part output display:

* **Top Row (Original Baseline):** Displays 4 identical side-by-side tiles of the pristine reference image ($512\times512$), providing a direct local comparison for each step.
* **Bottom Row (Distorted Grid):** Displays the corresponding distorted image right underneath its reference tile, visually highlighting how artifacts progress from mild to extreme.
* **Metric Trajectory Data Table:** Renders a Pandas DataFrame displaying the calculated numerical scores across all 5 metrics (**PSNR, SSIM, LPIPS-Alex, LPIPS-VGG, TPIPS**) alongside the distortion level.

---

#### 3. Execution Pipeline & Standardized Setup

1. **Resolution Normalization:** Input images are standardized to $512\times512$ float arrays ($[0.0, 1.0]$ range). This optimizes CPU execution speed for the 8B-parameter TPIPS backbone without sacrificing perceptual fidelity.
2. **Deterministic Evaluation:** Each code cell operates independently. Running a cell automatically applies the specific transformation function, visualizes the 2-row image matrix, computes all metric tensors, and outputs the formatted summary table.

In [ ]:
# =========================================================================
# TEST 1: (Clipped) Gaussian Noise Grid Sweep
# =========================================================================

# 1. Load and standardize reference image to 512x512
raw_img = cv2.imread("img_ref.jpeg")
if raw_img is None:
    raise FileNotFoundError("Could not find img_ref.jpeg in the current directory.")
raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
img_ref = cv2.resize(raw_img, (512, 512)) / 255.0


# 2. Define Noise Generator
def apply_gaussian_noise(img_np, sigma=10):
    """Adds zero-mean Gaussian noise scaled to float image range [0.0, 1.0]."""
    noise = np.random.normal(0, sigma / 255.0, img_np.shape)
    return np.clip(img_np + noise, 0.0, 1.0)


# 3. Run 2x4 visual grid sweep + metric summary table
df_noise = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_gaussian_noise,
    levels=[5, 10, 20, 40],
    param_name="Sigma (σ)",
    exp_title="Gaussian Noise Sensitivity",
)

In [ ]:
# =========================================================================
# TEST 2: Gaussian Blur Grid Sweep
# =========================================================================


def apply_gaussian_blur(img_np, radius=1):
    """Applies Gaussian Blur with odd kernel size derived from radius."""
    if radius <= 0:
        return img_np.copy()
    ksize = radius * 2 + 1
    return cv2.GaussianBlur(img_np, (ksize, ksize), 0)


# Run 2x4 visual grid + metric summary table
df_blur = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_gaussian_blur,
    levels=[1, 2, 4, 8],
    param_name="Radius",
    exp_title="Gaussian Blur",
)

In [ ]:
# =========================================================================
# TEST 3: JPEG Compression Grid Sweep
# =========================================================================

import io


def apply_jpeg_compression(img_np, quality=50):
    """Simulates JPEG compression artifacts in RAM."""
    img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
    buffer = io.BytesIO()
    img_pil.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    img_compressed = Image.open(buffer)
    return np.array(img_compressed) / 255.0


# Run 2x4 visual grid + metric summary table
df_jpeg = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_jpeg_compression,
    levels=[90, 70, 40, 10],
    param_name="Quality",
    exp_title="JPEG Compression",
)

In [ ]:
# =========================================================================
# TEST 4: Spatial Translation Grid Sweep
# =========================================================================


def apply_translation(img_np, shift_pixels=2):
    """Translates image horizontally with zero-padding at boundary."""
    rows, cols, _ = img_np.shape
    M = np.float32([[1, 0, shift_pixels], [0, 1, 0]])
    return cv2.warpAffine(img_np, M, (cols, rows))


# Run 2x4 visual grid + metric summary table
df_shift = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_translation,
    levels=[1, 2, 4, 8],
    param_name="Shift (px)",
    exp_title="Spatial Translation",
)

In [ ]:
# =========================================================================
# TEST 5: Brightness Shift Grid Sweep
# =========================================================================

from PIL import ImageEnhance


def apply_brightness(img_np, factor=1.2):
    """Scales image brightness."""
    img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
    enhancer = ImageEnhance.Brightness(img_pil)
    return np.array(enhancer.enhance(factor)) / 255.0


# Run 2x4 visual grid + metric summary table
df_bright = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_brightness,
    levels=[0.7, 0.9, 1.1, 1.3],
    param_name="Factor",
    exp_title="Brightness Shift",
)

In [ ]:
# =========================================================================
# TEST 6: Contrast Alterations
# =========================================================================


def apply_contrast(img_np, factor=1.0):
    """Scales image contrast."""
    img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
    return np.array(ImageEnhance.Contrast(img_pil).enhance(factor)) / 255.0


# Run 2x2 visual grid + metric summary table
df_contrast = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_contrast,
    levels=[0.7, 1.3],
    param_name="Factor",
    exp_title="Contrast Alterations",
)

In [ ]:
# =========================================================================
# TEST 7: Saturation Alterations
# =========================================================================


def apply_saturation(img_np, factor=1.0):
    """Scales image saturation (color intensity)."""
    img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
    return np.array(ImageEnhance.Color(img_pil).enhance(factor)) / 255.0


# Run 2x3 visual grid + metric summary table
df_saturation = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_saturation,
    levels=[0.0, 0.5, 1.5],
    param_name="Factor",
    exp_title="Saturation Alterations",
)

In [ ]:
# =========================================================================
# TEST 8: Spatial Rotation
# =========================================================================


def apply_rotation(img_np, degrees=0):
    """Rotates image around center with black boundary padding."""
    rows, cols, _ = img_np.shape
    M = cv2.getRotationMatrix2D((cols / 2, rows / 2), degrees, 1.0)
    return cv2.warpAffine(img_np, M, (cols, rows))


# Run 2x4 visual grid + metric summary table
df_rotation = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_rotation,
    levels=[1, 3, 5, 10],
    param_name="Degrees",
    exp_title="Spatial Rotation",
)

In [ ]:
# =========================================================================
# TEST 9: Crop + Resize
# =========================================================================


def apply_crop_resize(img_np, crop_percent=0):
    """Crops center region by percentage and resizes back to original size."""
    if crop_percent <= 0:
        return img_np.copy()
    h, w, _ = img_np.shape
    ch, cw = int(h * (crop_percent / 100.0)), int(w * (crop_percent / 100.0))
    cropped = img_np[ch : h - ch, cw : w - cw]
    return cv2.resize(cropped, (w, h))


# Run 2x4 visual grid + metric summary table
df_crop = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_crop_resize,
    levels=[2, 5, 10, 20],
    param_name="Crop %",
    exp_title="Crop + Resize",
)

In [ ]:
# =========================================================================
# TEST 10: Color Shift (Hue Offset)
# =========================================================================


def apply_color_shift(img_np, shift_deg=0):
    """Shifts color hue in HSV space."""
    img_hsv = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    img_hsv[:, :, 0] = (
        img_hsv[:, :, 0].astype(int) + int(shift_deg / 2)
    ) % 180
    shifted_rgb = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)
    return shifted_rgb / 255.0


# Run 2x4 visual grid + metric summary table
df_color_shift = plot_sweep_grid(
    img_ref=img_ref,
    distortion_fn=apply_color_shift,
    levels=[10, 30, 60, 90],
    param_name="Shift (°)",
    exp_title="Color Shift",
)

EXPERIMENT 2

In [ ]:
import glob
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Set your Google Drive Folder Path
# Update this path if the folder is located under "Shared with me" or a specific subfolder
DATASET_DIR = "/content/drive/MyDrive/1NQTRyU0nSjslrZocG2pTsXtCH2wNSGZh"

# Alternative path if it's in a shortcut/shared folder:
# DATASET_DIR = "/content/drive/MyDrive/Your_Folder_Name"

# 3. Load all image paths (JPEG, JPG, PNG)
valid_extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG")
image_paths = []
for ext in valid_extensions:
    image_paths.extend(glob.glob(os.path.join(DATASET_DIR, ext)))

image_paths = sorted(image_paths)
print(f"✅ Successfully loaded {len(image_paths)} images from Google Drive.")
for idx, path in enumerate(image_paths):
    print(f"  [Index {idx}] -> {os.path.basename(path)}")

Step 2: Dataset Evaluation Helper & Dual Graph Generator

In [ ]:
def run_dataset_experiment(
    image_paths,
    distortion_fn,
    levels,
    param_name="Level",
    exp_title="Dataset Experiment",
):
    """Processes all images in image_paths across distortion levels.

    Outputs aggregated statistics, a Dataset Summary Plot (Level vs Score), and
    a Per-Image Comparison Plot (Image Index vs Score).
    """
    all_results = []

    print(
        f"🚀 Running {exp_title} across {len(image_paths)} dataset images..."
    )

    for img_idx, img_path in enumerate(image_paths):
        # Load and resize image to 512x512
        raw_img = cv2.imread(img_path)
        if raw_img is None:
            print(f"⚠️ Warning: Could not load {img_path}, skipping...")
            continue
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        img_ref = cv2.resize(raw_img, (512, 512)) / 255.0

        for lvl in levels:
            img_dist = distortion_fn(img_ref, lvl)
            scores = compute_all_metrics(img_ref, img_dist)

            scores["Image_Index"] = img_idx
            scores["Image_Name"] = os.path.basename(img_path)
            scores[param_name] = lvl
            all_results.append(scores)

    df_all = pd.DataFrame(all_results)

    # -------------------------------------------------------------------------
    # PLOT 1: Dataset Summary Curve (Mean ± Std vs Distortion Level)
    # -------------------------------------------------------------------------
    df_summary = (
        df_all.groupby(param_name)
        .agg(
            psnr_mean=("PSNR (↑)", "mean"),
            psnr_std=("PSNR (↑)", "std"),
            ssim_mean=("SSIM (↑)", "mean"),
            ssim_std=("SSIM (↑)", "std"),
            lpips_alex_mean=("LPIPS-Alex (↓)", "mean"),
            lpips_alex_std=("LPIPS-Alex (↓)", "std"),
            tpips_mean=("TPIPS Distance (↓)", "mean"),
            tpips_std=("TPIPS Distance (↓)", "std"),
        )
        .reset_index()
    )

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(
        df_summary[param_name],
        df_summary["psnr_mean"],
        "r-o",
        label="PSNR (dB)",
        linewidth=2,
    )
    ax1.fill_between(
        df_summary[param_name],
        df_summary["psnr_mean"] - df_summary["psnr_std"],
        df_summary["psnr_mean"] + df_summary["psnr_std"],
        color="r",
        alpha=0.15,
    )
    ax1.set_xlabel(param_name, fontsize=11)
    ax1.set_ylabel("PSNR (↑)", color="r", fontsize=11)

    ax2 = ax1.twinx()
    ax2.plot(
        df_summary[param_name],
        df_summary["lpips_alex_mean"],
        "b-s",
        label="LPIPS-Alex",
        linewidth=2,
    )
    ax2.plot(
        df_summary[param_name],
        df_summary["tpips_mean"],
        "g-^",
        label="TPIPS Distance",
        linewidth=2,
    )
    ax2.set_ylabel("Perceptual Distance (↓)", color="b", fontsize=11)

    plt.title(f"{exp_title}: Dataset Aggregate (Mean ± Std)", fontsize=13)
    plt.show()

    # -------------------------------------------------------------------------
    # PLOT 2: Per-Image Sensitivity (Image Index vs Metrics at Max Level)
    # -------------------------------------------------------------------------
    max_level = levels[-1]
    df_max_lvl = df_all[df_all[param_name] == max_level]

    fig, ax1 = plt.subplots(figsize=(12, 4))

    ax1.plot(
        df_max_lvl["Image_Index"],
        df_max_lvl["PSNR (↑)"],
        "r-o",
        label="PSNR (dB)",
        linewidth=2,
    )
    ax1.set_xlabel(
        "Image Index (Matches Google Drive File Order)", fontsize=11
    )
    ax1.set_ylabel("PSNR (↑)", color="r", fontsize=11)
    ax1.set_xticks(df_max_lvl["Image_Index"])
    ax1.set_xticklabels(
        [f"Idx {i}: {name[:10]}..." for i, name in zip(df_max_lvl["Image_Index"], df_max_lvl["Image_Name"])],
        rotation=45,
        ha="right",
    )

    ax2 = ax1.twinx()
    ax2.plot(
        df_max_lvl["Image_Index"],
        df_max_lvl["LPIPS-Alex (↓)"],
        "b-s",
        label="LPIPS-Alex",
        linewidth=2,
    )
    ax2.plot(
        df_max_lvl["Image_Index"],
        df_max_lvl["TPIPS Distance (↓)"],
        "g-^",
        label="TPIPS Distance",
        linewidth=2,
    )
    ax2.set_ylabel("Perceptual Distance (↓)", color="b", fontsize=11)

    # Combine legends
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

    plt.title(
        f"{exp_title}: Per-Image Comparison at {param_name} = {max_level}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

    display(df_summary)
    return df_all

In [ ]:
# =========================================================================
# DATASET EXPERIMENT 1: Gaussian Noise across Drive Folder
# =========================================================================


def apply_gaussian_noise(img_np, sigma=10):
    noise = np.random.normal(0, sigma / 255.0, img_np.shape)
    return np.clip(img_np + noise, 0.0, 1.0)


df_dataset_noise = run_dataset_experiment(
    image_paths=image_paths,
    distortion_fn=apply_gaussian_noise,
    levels=[5, 10, 20, 40],
    param_name="Sigma (σ)",
    exp_title="Dataset Gaussian Noise Evaluation",
)